<!--

    Gaia Data Processing and Analysis Consortium (DPAC) 
    Co-ordination Unit 9 Work Package 930, based on 
    original scripts provided by the Apache SW Foundation
    
    (c) 2005-2025 Gaia DPAC
    
    This program is free software: you can redistribute it and/or modify
    it under the terms of the GNU General Public License as published by
    the Free Software Foundation, either version 3 of the License, or
    (at your option) any later version.

    This program is distributed in the hope that it will be useful,
    but WITHOUT ANY WARRANTY; without even the implied warranty of
    MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
    GNU General Public License for more details.

    You should have received a copy of the GNU General Public License
    along with this program.  If not, see <https://www.gnu.org/licenses/>.
    -->
    
This notebook illustrates a few tips and tricks that should aid users in interacting with the platform via the Jupyter user interface.

Users should be aware that the platform accesses a shared compute cluster. Depending on activity elsewhere cell execution may result in "Pending" jobs and we kindly request that users be patient. 

If, however, a running notebook cell becomes unresponsive (e.g. goes away running and never comes back) or behaves in unexpected ways (the Python interpreter can get occasionally tied in knots) as a last resort you can reset using the "Restart the kernel" option available in the menu above (the circular arrow). Note that this will kill all currently executing jobs in your context and free up all memory so you must re-establish the platform set-up in the resulting fresh Spark context by executing in PySpark

    # Import the spark config
    from astroflow_spark_gaia import spark

before doing anything else. (This is why we recommend that you include this line at the top of each notebook workflow).

You can export data from, or import data to the platform via your Jupyter account home directory (by default, `/home/jovyan/`) onto or from your local desktop. 

The easiest option is to find the data in the file explorer on your left (or click the file icon in the top left corner), and right-click on the files or directories you want to save, and select `Download`. Directories will be zipped for you. To import files, simply locate and click the `Upload Files` icon (upward pointing arrow) in the top left of the GUI.
    
To save in-memory data (for example as expressed in a results DataFrame) on the platform in a file for export, be aware that a Spark DataFrame is a distributed data set. If you save such an object to disk you will get a large set of partition files reflecting the natural distribution of the underlying source data. This is neither convenient nor particularly friendly. Provided the data size is not too large it is better to collect the distributed data to a non-distributed object on the master executor. The easiest way to do this is to call the `.toPandas()` method of the DataFrame, then this can be saved to a convenient format (e.g. comma-separated value). In the following simple example a DataFrame of the positions and magnitudes of all sources in the [Gaia Andromeda Photometric Survey (GAPS)](https://gea.esac.esa.int/archive/documentation/GDR3/Data_processing/chap_cu5pho/sec_cu5pho_gaps/) is created, collected to an intermediate non-distributed Pandas object, then saved to csv: 

In [3]:
# simple example of saving a results file to disk, e.g. prior to transfer off the platform to a user's local file system (see final paragraph in the description in the cell above)

# standard set-up
import gaiadmpsetup

# create an example data set - in this case a simple GAPS selection from gaia source
data_frame = spark.sql('SELECT ra, dec, parallax, parallax_error, phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag FROM gaia_source WHERE in_andromeda_survey')

# collect results to Pandas and save to csv in the user's home directory
# data_frame.toPandas().to_csv(path_or_buf = '/user/jovyan/gapscat.csv', index = False)


## Visualisation options

One big advantage of notebooks is that you can visualise data with your within your code and mark-down cells. [Matplotlib](https://matplotlib.org) is the premier Python plotting module available on this platform and it works in much the same way as other familiar Python environments (but note that an explicit call to `show()` is not necessary - plot rendering is accomplished via a post-execute hook which tells Jupyter to plot all currently open matplotlib figures after executing the rest of the paragraph). Saving a plot locally is as simple as calling the pyplot instance savefig() method (see above for instructions on transfering files off the platform).



In [14]:
import matplotlib.pyplot as plt

plt.plot([1,2,3,4])
plt.ylabel('some numbers')

# to save the plot file use the savefig method, substituting your username as appropriate:
# plt.savefig('/user/{YOUR-USERNAME}/somenums.png')

Pandas provides a high level api for visualisation of Pandas data frames. It uses matplotlib for its visualization under the hood, so the usage is the same as matplotlib. 

In [22]:
import pandas as pd
import numpy as np

ts = pd.Series(np.random.randn(1000), index=pd.date_range('1/1/2000', periods=1000))
ts = ts.cumsum()
ts.plot()

## Pandas User Defined Functions (a.k.a. vectorized UDFs)

A convenient feature of the PySpark SQL data frame API is the programmability afforded by user defined functions. Users who have found themselves limited by the small number of aggregate functions typically available in ADQL will find this feature particularly useful in scale-out usage scenarios. There are illustrations of the use of UDFs in the tutorial notebooks provided on this platform - see for example notebook 5, "Working with Gaia XP spectra". 

For further details see the [Apache Spark documentation for Pandas UDFs](https://spark.apache.org/docs/3.5.2/api/python/user_guide/sql/arrow_pandas.html?highlight=pandas%20udf#pandas-udfs-a-k-a-vectorized-udfs).

